# Code execution — analyze a CSV end to end

The **code execution** tool lets Claude write *and run* Python (plus Bash and file edits) inside
a secure sandbox on Anthropic's servers. Unlike the client-side tools elsewhere in this repo,
it's a **server tool**: Anthropic runs the code and hands you the results directly, so there's
*no tool-result loop for you to write*.

In this notebook we:

1. Run a quick warm-up calculation to see the tool fire.
2. Generate a deliberately messy sales CSV.
3. Upload it through the Files API.
4. Ask Claude to clean it, summarize it, and render a chart as a PNG.
5. Download the chart Claude created and display it here.

**Sandbox facts:** Python 3.11 on Linux, ~5 GiB RAM, **no internet access**, with `pandas`,
`numpy`, `matplotlib`, `scikit-learn` and friends preinstalled.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root).

## Setup

Helpers live in `_code_execution.py` (the tool definition, a `pause_turn`-aware request helper, response parsing, and the Files-API upload/download helpers), so the notebook stays focused on the flow.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "code_execution"):
    if os.path.isfile(os.path.join(_p, "_code_execution.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _code_execution import (
    CODE_EXECUTION_TOOL,
    SAMPLE_SALES_CSV,
    download_created_files,
    run_analysis,
    show_response,
    upload_file,
    write_sample_csv,
)

load_dotenv()
client = Anthropic()

## The tool definition

The tool takes no parameters — you just declare it. We use `code_execution_20250825`, which
supports Bash and file operations and works on every current model (including Haiku 4.5). When
it's present, Claude automatically gets two sub-tools behind the scenes: a `bash` runner and a
`text_editor` for creating/editing files.

> On Opus 4.5+ / Sonnet 4.5+, swap in `code_execution_20260120` for REPL state that persists
> across tool calls and programmatic tool calling from inside the sandbox.

In [ ]:
CODE_EXECUTION_TOOL

## Warm-up: a verified calculation

The simplest possible call. Claude decides on its own that the question is worth computing
rather than guessing, runs code, and reports the answer. Notice we never handle a `tool_use` /
`tool_result` exchange ourselves — the results just come back. `run_analysis` also transparently
resumes if the API returns `stop_reason == "pause_turn"` on a long job.

In [ ]:
response = run_analysis(
    client,
    "Compute the sample standard deviation of the first 50 prime numbers. Run code to be sure.",
    max_tokens=2048,
)
show_response(response)

## Step 1 — make a messy CSV

We write the file with the standard library only (no local pandas needed — the *sandbox* has
pandas). The data has the usual real-world problems: inconsistent region capitalization, missing
`units_sold`, missing `revenue` we can recompute, an exact duplicate row, and stray whitespace
in a product name.

In [ ]:
path = write_sample_csv("sample_sales.csv")
print("Wrote", path)
print(SAMPLE_SALES_CSV)

## Step 2 — upload through the Files API

To let the sandbox see your data, upload it and reference the returned file id. The Files API is
in beta, so `upload_file` uses `client.beta.*`; `run_analysis` sends the `files-api-2025-04-14`
beta header for us.

In [ ]:
file_object = upload_file(client, "sample_sales.csv")
file_object.id

## Step 3 — the flagship request

We reference the uploaded file with a `container_upload` content block alongside our
instructions. Claude loads the CSV, reasons about what's wrong, writes and runs code to clean
it, and saves a chart — all in one turn.

In [ ]:
analysis_prompt = (
    "You're given a messy sales CSV. Please:\n"
    "1. Load it and tell me what data-quality problems you find.\n"
    "2. Clean it: normalize the region names to title case, trim whitespace in product names, "
    "drop exact duplicate rows, and fill missing `revenue` as units_sold * unit_price. For rows "
    "missing units_sold, infer it from revenue and unit_price where possible, otherwise drop them.\n"
    "3. Show summary statistics of total revenue by region.\n"
    "4. Make a clean bar chart of total revenue by region and save it as `revenue_by_region.png`.\n"
    "Explain what you did at each step."
)

response = run_analysis(client, analysis_prompt, file_id=file_object.id, max_tokens=4096)
show_response(response)

## Step 4 — download the files Claude created

When Claude creates a file in the sandbox, its id rides along in the execution result blocks.
`download_created_files` scans the response for those ids and pulls each file down through the
Files API.

In [ ]:
saved = download_created_files(client, response)
saved

Display the chart Claude rendered:

In [ ]:
from IPython.display import Image

Image(filename="revenue_by_region.png")

## Notes

- **Pricing.** Code execution is *free* when `web_search` or `web_fetch` is in the same request.
  Otherwise it's billed by execution time — each org gets free hours per month (5-minute minimum
  per session), then a per-hour-per-container rate. Attaching files preloads the container, so
  time is billed even if Claude never runs code.
- **Stateful sessions.** Pass `container=<previous response>.container.id` (via the `container=`
  argument of `run_analysis`) to reuse the same sandbox and keep files between calls.
- **No internet.** The sandbox can't make outbound requests, so anything Claude needs must be
  uploaded or generated in-container.
- **Tool version.** `code_execution_20250825` is the safe default; `code_execution_20260120`
  adds persistent REPL state and programmatic tool calling on Opus 4.5+ / Sonnet 4.5+.